In [ ]:
print("Notebook working")

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500  # number of employees

data = pd.DataFrame({
    "age": np.random.randint(22, 60, n),
    "experience_years": np.random.randint(0, 20, n),
    "department": np.random.choice(["HR", "IT", "Sales", "Finance"], n),
    "salary": np.random.randint(20000, 150000, n),
    "training_hours": np.random.randint(0, 100, n),
    "projects_completed": np.random.randint(0, 20, n),
    "attendance_rate": np.random.uniform(0.5, 1.0, n),
    "manager_feedback": np.random.randint(1, 6, n),
    "overtime_hours": np.random.randint(0, 50, n)
})

In [ ]:
def assign_performance(row):
    score = 0
    
    score += row["projects_completed"] * 0.3
    score += row["attendance_rate"] * 10
    score += row["manager_feedback"] * 2
    score += row["training_hours"] * 0.05
    
    if row["overtime_hours"] > 30:
        score -= 2
    
    if score > 25:
        return "High"
    elif score > 18:
        return "Medium"
    else:
        return "Low"

data["performance_rating"] = data.apply(assign_performance, axis=1)

In [ ]:
data.to_csv("../data/employee_data.csv", index=False)

In [ ]:
data.head()
data["performance_rating"].value_counts()

In [ ]:
 data["performance_rating"].value_counts(normalize=True)

In [ ]:
                                   # CELL 1

import pandas as pd
import numpy as np

In [ ]:
# CELL 2

data = pd.read_csv("../data/employee_data.csv")

print("Shape:", data.shape)
data.head()

In [ ]:
# CELL 3

data.info()

In [ ]:
# CELL 4

data.isnull().sum()

In [ ]:
# CELL 5

data["performance_rating"].value_counts()
data["performance_rating"].value_counts(normalize=True)

In [ ]:
# CELL 6

data.columns = data.columns.str.lower().str.replace(" ", "_")
data.columns

In [ ]:
# CELL 7

data.dtypes

In [ ]:
# CELL 8

# Numeric
data.fillna(data.median(numeric_only=True), inplace=True)

# Categorical
data.fillna("Unknown", inplace=True)

data.isnull().sum()

In [ ]:
# CELL 9

data = data.drop_duplicates()

print("New Shape:", data.shape)

In [ ]:
# CELL 10

data.describe()

In [ ]:
# CELL 11

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [ ]:
# CELL 12

sns.countplot(x="performance_rating", data=data)
plt.title("Performance Rating Distribution")
plt.show()

In [ ]:
# CELL 13

sns.boxplot(x="performance_rating", y="age", data=data)
plt.title("Age vs Performance")
plt.show()

In [ ]:
# CELL 14

sns.boxplot(x="performance_rating", y="salary", data=data)
plt.title("Salary vs Performance")
plt.show()

In [ ]:
# CELL 15

sns.boxplot(x="performance_rating", y="training_hours", data=data)
plt.title("Training Hours vs Performance")
plt.show()

In [ ]:
# CELL 16

sns.boxplot(x="performance_rating", y="attendance_rate", data=data)
plt.title("Attendance vs Performance")
plt.show()# CELL 16


In [ ]:
# CELL 17

plt.figure(figsize=(10,6))
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

In [ ]:
# CELL 18

X = data.drop("performance_rating", axis=1)
y = data["performance_rating"]

print(X.shape, y.shape)

In [ ]:
# CELL 19

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

In [ ]:
# CELL 20

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# CELL 21

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# CELL 22

# Numeric pipeline
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [ ]:
# CELL 23

X_train_processed = preprocessor.fit_transform(X_train)

print("Processed shape:", X_train_processed.shape)

In [ ]:
# CELL 24

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# CELL 25

log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=200, class_weight="balanced"))
])

log_model.fit(X_train, y_train)

In [ ]:
# CELL 26

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Results:\n")
print(classification_report(y_test, y_pred_log))

In [ ]:
# CELL 27

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_log))

In [ ]:
# CELL 28

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

In [ ]:
# CELL 29

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results:\n")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# CELL 30

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
!pip install imbalanced-learn

In [ ]:
import sys
!{sys.executable} -m pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_smote.value_counts())

In [ ]:
from sklearn.linear_model import LogisticRegression

model_smote = LogisticRegression(max_iter=1000)

model_smote.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_smote = model_smote.predict(X_test_processed)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_smote = model_smote.predict(X_test_processed)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_smote))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_rf = rf_model.predict(X_test_processed)

In [ ]:
from sklearn.metrics import classification_report

print("Random Forest Results:\n")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_rf)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['High', 'Low', 'Medium'],
            yticklabels=['High', 'Low', 'Medium'])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
import pandas as pd

feature_importance = rf_model.feature_importances_

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importance
})

importance_df = importance_df.sort_values(by="importance", ascending=False)

print(importance_df.head(10))

In [ ]:
import matplotlib.pyplot as plt

top_features = importance_df.head(10)

plt.figure()
plt.barh(top_features["feature"], top_features["importance"])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importance")
plt.xlabel("Importance")
plt.show()

In [ ]:
import joblib

# save model
joblib.dump(model_smote, "../models/model.pkl")

print("✅ Model saved successfully!")

In [ ]:
joblib.dump(pipeline_smote, "../models/model.pkl")

In [1]:
# ===== FULL TRAIN + SAVE (ONE CELL) =====

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Load data
data = pd.read_csv("../data/employee_data.csv")

# Features & target
X = data.drop("performance_rating", axis=1)
y = data["performance_rating"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Columns
cat_cols = ["department"]
num_cols = [col for col in X.columns if col != "department"]

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# FINAL PIPELINE (SMOTE + MODEL)
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Save model
joblib.dump(pipeline, "../models/model.pkl")

print("✅ MODEL TRAINED & SAVED SUCCESSFULLY")

✅ MODEL TRAINED & SAVED SUCCESSFULLY
